# சவால்: தரவு அறிவியல் பற்றிய உரையைப் பகுத்தறிதல்

இந்த எடுத்துக்காட்டில், பாரம்பரிய தரவு அறிவியல் செயல்முறையின் அனைத்து படிகளையும் 포함ும் எளிய பயிற்சியை செய்வோம். நீங்கள் எந்தக் கோடியையும் எழுத தேவையில்லை, கீழுள்ள செல்லுகளைச் சொடுக்கி அவற்றை செயல்படுத்தி அதின் விளைவைக் காணலாம். சவாலாக, நீங்கள் இந்த கோடைக் வேறு தரவுகளுடன் முயற்சி செய்ய ஊக்கப்படுகிறீர்கள்.

## குறிக்கோள்

இந்த பாடத்தில், நாங்கள் தரவு அறிவியலைச் சார்ந்த வெவ்வேறு ধারণைகள் பற்றி பேசிக் கொண்டிருக்கிறோம். **உரைக் கண்காணிப்பு** செய்து மேலும் தொடர்புடைய கருத்துக்களை கண்டறிய முயற்சிப்போமாம். தரவு அறிவியல் பற்றிய ஒரு உரையால் துவங்கி, அதிலிருந்து முக்கிய வார்த்தைகளை எடுத்து, பிறகு அதன் முடிவுகளை காட்சி படுத்த முயற்சிப்போம்.

ஒரு உரையாக, நான் விக்கிப்பீடியாவில் தரவு அறிவியல் பக்கம் பயன்படுத்தப்போகிறேன்:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## படி 1: தரவைப் பெறுதல்

ஒவ்வொரு தரவியல் அறிவியல் செயல்முறையின் முதல் படி தரவைப் பெறுதல். அதற்கு நாம் `requests` நூலகத்தை பயன்படுத்தப்போகிறோம்:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## படி 2: தரவை மாற்றல்

அடுத்த படி, தரவை செயலாக்கத்திற்கு ஏற்ற வடிவுக்கு மாற்றுவதே ஆகும். எங்கள் மாம்பழத்தில், நாங்கள் பக்கத்திலிருந்து HTML மூலக் குறியீட்டை பதிவிறக்கம் செய்துள்ளோம், அதை சாதாரண உரையாக மாற்ற வேண்டும்.

இதை செய்ய பல வழிகள் உள்ளன. நாம் [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/) என்ற பிரபலமான Python நூலகத்தை பயன்படுத்த விரும்புகிறோம், இது HTMLஐ பாகுபடுத்த உதவும். BeautifulSoup நமக்குப் புறக்கணிக்கக்கூடிய சில வழிசெலுத்தல் மெனுக்கள், பக்கவாசிகள், அடிக்குறிப்புகள் மற்றும் மற்ற பொருத்தமற்ற உள்ளடக்கங்களை குறைத்துவிட்டு, விக்கிப்பீடியாவின் முக்கிய கட்டுரையான உள்ளடக்கத்தை மட்டுமே கவனிக்க HTML கூறுகளைத் தேர்ந்தெடுக்க அனுமதிக்கிறது (எங்கேனும் சில பொதுவான உரை இன்னும் இருக்கும்).


முதலில், HTML பார்சிங்குக்காக BeautifulSoup நூலகத்தை நிறுவ வேண்டும்:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## படி 3: بصیرتங்களைப் பெறுதல்

மிகவும் முக்கியமான படி எங்கள் தரவை ஒரு வடிவமாக மாற்றுவது, அதிலிருந்து நாம் بصیرتங்களை எடுத்துக்கொள்ள முடியும். எங்கள் நிலையில், உரையிலிருந்து முக்கியத்துவமான சொற்களை எடுப்பதைக் குறிக்க விரும்புகிறோம், மேலும் எந்த முக்கியத்துவமான சொற்கள் அதிக அர்த்தமுள்ளதாக இருக்கின்றன என்பதை பார்க்க விரும்புகிறோம்.

முக்கிய சொற்கள் எடுப்பதற்கு Python நூலகமான [RAKE](https://github.com/aneesha/RAKE) ஐ நாங்கள் பயன்படுத்துவோம். முதலில், இந்த நூலகம் இல்லாவிட்டால் அதை நிறுவுவோம்:


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

முக்கிய அம்சம் `Rake` பொருள் மூலம் கிடைக்கிறது, இதனை நாம் சில அளவுருக்களை பயன்படுத்தி விருப்பப்படுத்தலாம். எங்களின் உட்பிரிவில், முக்கிய வார்த்தையின் குறைந்தபட்ச நீளம் 5 எழுத்துகள், ஆவணத்தில் ஒரு முக்கிய வார்த்தையின் குறைந்தபட்ச குறும்பாடல் 3 மற்றும் முக்கிய வார்த்தையில் அதிகபட்ச வார்த்தைகள் எண்ணிக்கை 2 ஆக அமைக்கப்படும். பிற மதிப்புகளுடன் சுதந்திரமாக செயல்பட்டு முடிவைப் பாருங்கள்.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


நாங்கள் தொடர்புடைய முக்கியத்துவம் அளவிடப்பட்ட பட்டியலுடன் ஒரு சொற்கள் பட்டியலைப் பெற்றுள்ளோம். நீங்கள் காணலாம், இயந்திரக் கற்றல் மற்றும் பெரிய தரவு போன்ற மிகவும் தொடர்புடைய துறைமுகங்கள் பட்டியலில் மேல்தள இடங்களில் உள்ளன.

## படி 4: முடிவை காட்சி வடிவில் காண்பித்தல்

மக்கள் தரவுகளை காட்சி வடிவில் சிறந்த முறையில் பொருளா புரிந்து கொள்ள முடியும். எனவே சில சமயங்களில் தரவை காட்சி வடிவில் காண்பிப்பது சில அறிவுரைகளை பெறுவதற்கு பொருத்தமாக இருக்கும். உள்ளடக்க முக்கியத்துவத்துடன் கூடிய முக்கியமுள்ள சொல்ல்களின் எளிமையான பகிர்வை வரைபடம் வரைய `matplotlib` நூலகத்தை Python இல் பயன்படுத்தலாம்:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

அப்படியென்றாலும், சொல் հաճախமைகளை காண்பிப்பதற்காக இன்னும் சிறந்த வழி ஒன்று உண்டு - **சொல் மேகம்** பயன்படுத்துவது. எங்கள் முக்கிய சொல் பட்டியலில் இருந்து சொல் மேகத்தை வரைப்பதற்கு நமக்கு இன்னொரு நூலகத்தை நிறுவ வேண்டும்.


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud` பொருள் மூலம் இயல்புநிலையான உரை அல்லது வார்த்தைகளின் எண்ணிக்கையுடன் முன்பே கணக்கிடப்பட்ட பட்டியலை ஏற்று, பின்னர் `matplotlib` ஐ பயன்படுத்தி காண்பிக்கக்கூடிய படத்தை 반환ிக்கிறது:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

நாமும் `WordCloud` க்கு மெய்நிகர் உரையை அனுப்ப முடியும் - விளைவாக ஒரே மாதிரி முடிவைப் பெற முடியும் என பார்த்துச் செய்போம்:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

இந்த வார்த்தை மேகம் இப்போது மேலும் பிரமாதமாக தோன்றுகிறது, ஆனால் அதிலும் அதிகமான சத்தம் உள்ளது (உதா. தொடர்பற்ற வார்த்தைகள் kuten `Retrieved on`). மேலும், *data scientist*, அல்லது *computer science* போன்ற இரண்டு வார்த்தைகள் கொண்ட குறைந்த முக்கிய வார்த்தைகள் கிடைக்கின்றன. இது RAKE அல்காரிதம் உரையிலிருந்து நல்ல முக்கிய வார்த்தைகளை தேர்ந்தெடுப்பதில் மிகச் சிறப்பாக செயல்படுவதால். இந்த உதாரணம் தரவுகளின் முன்கூட்டிய செயலாக்கமும் தூய்மையும் முக்கியத்துவத்தை விளக்குகிறது, ஏனெனில் கடைசியில் தெளிவான படம் சிறந்த முடிவுகளை எடுக்க உதவும்.

இந்த பயிற்சியில், விக்கிப்பீடியா உரையிலிருந்து முக்கியத்துவம் கொண்ட எழுத்துகளை மற்றும் வார்த்தை மேகத்தை எடுக்க எளிய முறையைப் பார்த்தோம். இந்த உதாரணம் மிகவும் எளிமையானது, ஆனால் தரவாளரும் தரவுகளுடன் பணியாற்றும்போது எடுக்கும் அனைத்து முதன்மையான படிகளையும் நன்றாக விளக்குகிறது, தரவுத் தொகுப்பிலிருந்து தொடங்கி, காட்சி வடிவமைப்புவரை.

நமது பாடத்தில் அந்த அனைத்து படிகளையும் விரிவாகப் பேசப்போகின்றோம்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
